<a href="https://colab.research.google.com/github/kiryu-arai/kaggle_compedition_monster/blob/suzuki/v5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# カリフォルニア住宅価格予測 - 発展的な特徴量エンジニアリングと交差検証パイプライン

このノートブックでは、以下の5つのアプローチを統合した高度なパイプラインを実行します。
1. **PCA（主成分分析）** によるサイズ関連変数の次元縮約
2. **主要都市（SF/LA）からの距離** の算出による地理情報の集約
3. **K-Meansクラスタリング** による地域ブロックの自動グループ化
4. **5-Fold 交差検証（Cross Validation）** による頑健な評価とアンサンブル予測
5. **目的変数（Price）の対数変換（log1p/expm1）** による予測精度の安定化

5/25深夜
parameter_set : {'learning_rate': 0.03710885609231727, 'depth': 8, 'l2_leaf_reg': 7.772874592555051, 'bagging_temperature': 0.3217767825180894}
Best CatBoost CV (RMSE): 0.4239
⚠️ 16 trials 到達したが未達成 (best: 0.4239)
XGB OOF RMSE: 0.4105
parameter_set : {'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.602181434610246, 'device': 'cuda', 'early_stopping_rounds': 50, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.01661220560195241, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 8, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 5000, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': 0.0023806589316786608, 'reg_lambda': 0.028903948082561075, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.9157632062844006, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}
CatBoost OOF RMSE: 0.4106
parameter_set : {'iterations': 5000, 'learning_rate': 0.03710885609231727, 'depth': 8, 'l2_leaf_reg': 7.772874592555051, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': 0, 'bagging_temperature': 0.3217767825180894, 'cat_features': [88]}

深夜2
parameter_set : {'learning_rate': 0.09867641758685206, 'depth': 8, 'l2_leaf_reg': 9.123241431830145, 'bagging_temperature': 0.36534542475167564}
Best CatBoost CV (RMSE): 0.4219
⚠️ 16 trials 到達したが未達成 (best: 0.4219)
XGB OOF RMSE: 0.4105
parameter_set : {'objective': 'reg:squarederror', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.602181434610246, 'device': 'cuda', 'early_stopping_rounds': 50, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': None, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.01661220560195241, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 'max_delta_step': None, 'max_depth': 8, 'max_leaves': None, 'min_child_weight': None, 'missing': nan, 'monotone_constraints': None, 'multi_strategy': None, 'n_estimators': 5000, 'n_jobs': None, 'num_parallel_tree': None, 'random_state': 42, 'reg_alpha': 0.0023806589316786608, 'reg_lambda': 0.028903948082561075, 'sampling_method': None, 'scale_pos_weight': None, 'subsample': 0.9157632062844006, 'tree_method': 'hist', 'validate_parameters': None, 'verbosity': None}
CatBoost OOF RMSE: 0.4154
parameter_set : {'iterations': 5000, 'learning_rate': 0.09867641758685206, 'depth': 8, 'l2_leaf_reg': 9.123241431830145, 'loss_function': 'RMSE', 'random_seed': 42, 'verbose': 0, 'bagging_temperature': 0.36534542475167564, 'cat_features': [88]}

In [ ]:
!pip install optuna -q
!pip install catboost -q

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
  for filename in filenames:
    print(os.path.join(dirname, filename))

/kaggle/input/competitions/ambl-california-housing/sample.csv
/kaggle/input/competitions/ambl-california-housing/train.csv
/kaggle/input/competitions/ambl-california-housing/test.csv


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings; warnings.filterwarnings('ignore')

train = pd.read_csv('/kaggle/input/competitions/ambl-california-housing/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/ambl-california-housing/test.csv')
sample = pd.read_csv('/kaggle/input/competitions/ambl-california-housing/sample.csv')

print(f'Train: {train.shape}  Test: {test.shape}')
train.head()

Train: (16512, 13)  Test: (4128, 12)


,id,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Household,AllRooms,AllBedrms,Price
0,0,1.4817,6.0,4.443645,1.134293,1397.0,3.350120,36.77,-119.84,417.0,1853.0,473.0,0.720
1,1,6.9133,8.0,5.976471,1.026471,862.0,2.535294,33.68,-117.80,340.0,2032.0,349.0,2.741
2,2,1.5536,25.0,4.088785,1.000000,931.0,4.350467,36.60,-120.19,214.0,875.0,214.0,0.583
3,3,1.5284,31.0,2.740088,1.008811,597.0,2.629956,34.10,-118.32,227.0,622.0,229.0,2.000
4,4,4.0815,21.0,5.166667,1.002688,1130.0,3.037634,37.79,-121.23,372.0,1922.0,373.0,1.179


### モジュールの準備

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = '/kaggle/input/competitions/ambl-california-housing'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample = pd.read_csv(f'{DATA_DIR}/sample.csv')

print(f'Train shape: {train.shape} | Test shape: {test.shape}')

Train shape: (16512, 13) | Test shape: (4128, 12)


### 特徴量エンジニアリング（FE）の定義と実行

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree

def haversine_distance(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return c * 6371

def add_features(train_df, test_df):
    X_tr = train_df.copy()
    X_te = test_df.copy()

    y_tr = np.log1p(X_tr['Price'])
    X_tr = X_tr.drop(columns=['Price'])

    combined = pd.concat([X_tr, X_te], axis=0).reset_index(drop=True)
    lats    = combined['Latitude'].values
    lons    = combined['Longitude'].values
    n_train = len(X_tr)

    # 1. 比率・相互作用特徴量
    combined['rooms_per_person']     = combined['AveRooms']  / (combined['AveOccup']  + 1e-3)
    combined['bedrooms_ratio']       = combined['AveBedrms'] / (combined['AveRooms']  + 1e-3)
    combined['income_per_room']      = combined['MedInc']    / (combined['AveRooms']  + 1e-3)
    combined['rooms_per_household']  = combined['AllRooms']  / (combined['Household'] + 1e-3)
    combined['bedrms_per_household'] = combined['AllBedrms'] / (combined['Household'] + 1e-3)
    combined['total_income_proxy']   = combined['MedInc']    *  combined['Household']
    combined['nonbedroom_rooms']     = combined['AveRooms']  -  combined['AveBedrms']
    combined['household_density']    = combined['Household'] / (combined['AllRooms']  + 1e-3)

    # 2. 主要都市距離
    cities = {
        'LA': (34.0522, -118.2437), 'SF': (37.7749, -122.4194),
        'SD': (32.7157, -117.1611), 'SJ': (37.3382, -121.8863),
        'SB': (34.4208, -119.6982),
    }
    for city, (clat, clon) in cities.items():
        combined[f'dist_{city}'] = haversine_distance(lats, lons, clat, clon)
    dist_cols = [f'dist_{c}' for c in cities]
    combined['dist_nearest_city'] = combined[dist_cols].min(axis=1)
    combined['coast_dist_proxy']  = combined[['dist_SF', 'dist_LA', 'dist_SD']].min(axis=1)

    # 3. 名門大学インデックス
    df_univ = pd.DataFrame({
        'lat': [37.8719, 37.4275, 34.1377, 34.0689, 34.0224, 38.5382, 32.8801, 34.4140],
        'lon': [-122.2585, -122.1697, -118.1253, -118.4452, -118.2851, -121.7617, -117.2340, -119.8489],
        'education_brand_score': [10, 10, 10, 9, 8, 8, 9, 8]
    })
    univ_tree = KDTree(df_univ[['lat', 'lon']].values)
    univ_dists, univ_idx = univ_tree.query(combined[['Latitude', 'Longitude']].values, k=1)
    combined['dist_to_elite_univ']       = univ_dists.flatten()
    combined['nearest_univ_brand_score'] = df_univ['education_brand_score'].iloc[univ_idx.flatten()].values
    combined['univ_influence_score']     = combined['nearest_univ_brand_score'] / (combined['dist_to_elite_univ'] + 1e-2)

    # 4. 1990年治安リスクインデックス
    df_crime = pd.DataFrame({
        'lat': [34.0116, 34.0522, 33.8958, 37.8044, 37.9358, 36.7378, 38.5816, 37.7749, 32.7157, 33.6846, 37.4419, 37.9735],
        'lon': [-118.2741,-118.2437,-118.2201,-122.2712,-122.3477,-119.7871,-121.4944,-122.4194,-117.1611,-117.8265,-122.1430,-122.5311],
        'crime_severity_index': [10.0, 9.5, 9.8, 9.6, 8.5, 7.0, 6.5, 7.5, 4.5, 1.2, 1.5, 1.8]
    })
    crime_tree = KDTree(df_crime[['lat', 'lon']].values)
    crime_dists, crime_idx = crime_tree.query(combined[['Latitude', 'Longitude']].values, k=1)
    combined['nearest_crime_idx_1990']  = df_crime['crime_severity_index'].iloc[crime_idx.flatten()].values
    combined['dist_to_crime_benchmark'] = crime_dists.flatten()
    combined['local_crime_exposure']    = combined['nearest_crime_idx_1990'] / (combined['dist_to_crime_benchmark'] + 1e-2)

    # 5. 地震リスク: San Andreas断層 + Hayward断層への距離
    SAN_ANDREAS = np.array([
        [32.6,-115.5],[33.0,-115.8],[33.5,-116.3],[34.0,-116.8],[34.5,-117.8],
        [35.0,-118.8],[35.5,-120.0],[36.0,-120.5],[36.5,-121.0],[37.0,-121.5],
        [37.5,-121.9],[38.0,-122.2],[38.5,-122.7],[39.0,-123.1],[40.0,-124.0],
    ])
    HAYWARD = np.array([
        [37.3,-121.9],[37.5,-122.0],[37.7,-122.1],[37.8,-122.15],[37.9,-122.2],[38.0,-122.3],
    ])
    sa_d = np.column_stack([haversine_distance(lats, lons, p[0], p[1]) for p in SAN_ANDREAS])
    hw_d = np.column_stack([haversine_distance(lats, lons, p[0], p[1]) for p in HAYWARD])
    combined['dist_san_andreas']   = sa_d.min(axis=1)
    combined['dist_hayward_fault'] = hw_d.min(axis=1)
    combined['min_fault_dist']     = np.minimum(combined['dist_san_andreas'], combined['dist_hayward_fault'])
    combined['seismic_risk_score'] = 1.0 / (combined['min_fault_dist'] + 1.0)

    # 6. 軍事基地
    MILITARY_BASES = {
        'NAS_North_Island': (32.70,-117.21), 'NAS_Alameda':      (37.79,-122.30),
        'Travis_AFB':       (38.27,-121.93), 'Edwards_AFB':      (34.90,-117.88),
        'Vandenberg_AFB':   (34.74,-120.57), 'Fort_Ord':         (36.65,-121.76),
        'MCAS_El_Toro':     (33.67,-117.73), 'Point_Mugu':       (34.12,-119.12),
        'Castle_AFB':       (37.38,-120.57), 'NAS_Miramar':      (32.87,-117.14),
    }
    mil_cols = []
    for base, (blat, blon) in MILITARY_BASES.items():
        col = f'dist_mil_{base}'
        combined[col] = haversine_distance(lats, lons, blat, blon)
        mil_cols.append(col)
    combined['dist_nearest_military'] = combined[mil_cols].min(axis=1)

    # 7. 港湾アクセス
    PORTS = {
        'LA_LB':   (33.74,-118.27), 'Oakland': (37.80,-122.28), 'SD': (32.73,-117.17),
    }
    port_cols = []
    for port, (plat, plon) in PORTS.items():
        col = f'dist_port_{port}'
        combined[col] = haversine_distance(lats, lons, plat, plon)
        port_cols.append(col)
    combined['dist_nearest_port'] = combined[port_cols].min(axis=1)

    # 8. シリコンバレー雇用圏
    combined['dist_silicon_valley'] = haversine_distance(lats, lons, 37.39, -122.05)

    # 9. メキシコ国境
    combined['dist_mexico_border'] = haversine_distance(lats, lons, 32.54, -117.03)

    # 10. 高所得住宅地への引力
    AFFLUENT = {
        'Beverly_Hills': (34.07,-118.40), 'Palo_Alto':     (37.44,-122.15),
        'Marin':         (37.95,-122.55), 'Newport_Beach': (33.62,-117.93),
        'Santa_Barbara': (34.42,-119.70),
    }
    aff_cols = []
    for area, (alat, alon) in AFFLUENT.items():
        col = f'dist_aff_{area}'
        combined[col] = haversine_distance(lats, lons, alat, alon)
        aff_cols.append(col)
    combined['dist_nearest_affluent'] = combined[aff_cols].min(axis=1)
    combined['affluent_gravity']      = 1.0 / (combined['dist_nearest_affluent'] + 1.0)

    # 11. 経済地帯フラグ
    combined['zone_bay_area'] = (
        (combined['Latitude']  > 37.0) & (combined['Latitude']  < 38.5) &
        (combined['Longitude'] > -122.8) & (combined['Longitude'] < -121.4)
    ).astype(int)
    combined['zone_la_basin'] = (
        (combined['Latitude']  > 33.6) & (combined['Latitude']  < 34.5) &
        (combined['Longitude'] > -118.8) & (combined['Longitude'] < -117.5)
    ).astype(int)
    combined['zone_san_diego']       = (combined['Latitude'] < 33.2).astype(int)
    combined['zone_central_valley']  = (
        (combined['Longitude'] > -121.8) & (combined['Longitude'] < -118.5) &
        (combined['Latitude']  > 35.0)   & (combined['Latitude']  < 38.5)
    ).astype(int)
    combined['zone_inland_empire']   = (
        (combined['Latitude']  > 33.8) & (combined['Latitude']  < 34.3) &
        (combined['Longitude'] > -117.8) & (combined['Longitude'] < -116.9)
    ).astype(int)
    combined['zone_orange_county']   = (
        (combined['Latitude']  > 33.4) & (combined['Latitude']  < 34.0) &
        (combined['Longitude'] > -118.0) & (combined['Longitude'] < -117.4)
    ).astype(int)

    # 12. NorCal/SoCal分断
    combined['is_norcal'] = (combined['Latitude'] > 35.5).astype(int)
    combined['is_socal']  = (combined['Latitude'] < 34.5).astype(int)

    # 13. Kern County石油地帯
    combined['dist_kern_oil'] = haversine_distance(lats, lons, 35.37, -119.02)

    # 14. 精密海岸線距離
    COAST = np.array([
        [41.75,-124.20],[41.00,-124.15],[40.50,-124.10],[39.80,-123.95],[39.30,-123.80],
        [38.80,-123.50],[38.30,-123.10],[37.80,-122.65],[37.50,-122.50],[37.20,-122.40],
        [36.95,-122.05],[36.50,-121.95],[36.20,-121.75],[35.65,-121.30],[35.30,-120.85],
        [34.85,-120.65],[34.45,-120.00],[34.20,-119.30],[34.05,-118.80],[34.05,-118.50],
        [33.85,-118.40],[33.55,-118.00],[33.20,-117.40],[32.85,-117.25],[32.65,-117.15],[32.55,-117.10],
    ])
    coast_d = np.column_stack([haversine_distance(lats, lons, cp[0], cp[1]) for cp in COAST])
    combined['coast_dist_precise'] = coast_d.min(axis=1)

    # 15. 所得×地域フラグの交互作用
    combined['income_x_bay']    = combined['MedInc'] * combined['zone_bay_area']
    combined['income_x_la']     = combined['MedInc'] * combined['zone_la_basin']
    combined['income_x_norcal'] = combined['MedInc'] * combined['is_norcal']

    # 16. 郡レベル教育・貧困・失業スコア
    COUNTY_DATA = pd.DataFrame({
        'county': [
            'San Francisco','Marin','San Mateo','Santa Clara','Alameda',
            'Contra Costa','Sonoma','Napa','Solano','Sacramento',
            'Yolo','Placer','El Dorado','Los Angeles','Orange',
            'San Diego','Ventura','Santa Barbara','San Luis Obispo','Monterey',
            'Santa Cruz','San Joaquin','Stanislaus','Merced','Fresno',
            'Tulare','Kings','Kern','San Bernardino','Riverside','Imperial',
        ],
        'lat': [
            37.77, 37.95, 37.56, 37.33, 37.65, 37.92, 38.44, 38.29, 38.27, 38.58,
            38.69, 38.89, 38.77, 34.05, 33.72, 32.86, 34.27, 34.70, 35.55, 36.24,
            36.97, 37.94, 37.56, 37.19, 36.78, 36.22, 36.10, 35.37, 34.10, 33.95, 33.04,
        ],
        'lon': [
            -122.42,-122.55,-122.31,-121.89,-121.87,-122.00,-122.72,-122.28,-122.00,-121.49,
            -121.90,-120.80,-120.52,-118.24,-117.83,-117.10,-119.23,-119.70,-120.65,-121.60,
            -121.98,-121.27,-121.00,-120.72,-119.79,-119.00,-119.82,-119.02,-117.29,-117.39,-115.37,
        ],
        'edu_score':     [9,10,9,10,8,8,7,7,6,7,9,8,8,6,8,8,8,8,8,6,8,5,5,4,4,3,3,4,5,5,3],
        'poverty_score': [4,2,2,2,4,3,3,3,4,5,4,2,2,6,3,4,3,4,3,6,4,7,7,8,8,9,9,8,7,7,10],
        'unemp_score':   [3,2,2,2,4,3,4,3,5,5,4,3,3,6,3,4,3,4,4,5,4,7,6,8,8,8,8,7,6,6,9],
    })
    county_tree = KDTree(COUNTY_DATA[['lat', 'lon']].values)
    _, county_idx = county_tree.query(combined[['Latitude', 'Longitude']].values, k=1)
    county_idx = county_idx.flatten()
    combined['county_edu_score']    = COUNTY_DATA['edu_score'].iloc[county_idx].values
    combined['county_poverty_score']= COUNTY_DATA['poverty_score'].iloc[county_idx].values
    combined['county_unemp_score']  = COUNTY_DATA['unemp_score'].iloc[county_idx].values
    combined['county_quality_idx']  = (
        combined['county_edu_score']
        - 0.6 * combined['county_poverty_score']
        - 0.4 * combined['county_unemp_score']
    )
    combined['income_x_county_edu'] = combined['MedInc'] * combined['county_edu_score']

    # 17. BART駅への距離
    BART_STATIONS = np.array([
        [37.937,-122.353],[37.925,-122.317],[37.902,-122.299],[37.874,-122.283],
        [37.870,-122.268],[37.853,-122.270],[37.829,-122.267],[37.808,-122.269],
        [37.804,-122.272],[37.798,-122.265],[37.775,-122.224],[37.754,-122.198],
        [37.722,-122.161],[37.697,-122.127],[37.670,-122.088],[37.635,-122.057],
        [37.591,-122.017],[37.557,-121.977],
        [37.973,-122.029],[37.948,-122.057],[37.905,-122.067],[37.893,-122.124],
        [37.878,-122.183],[37.844,-122.251],[37.805,-122.295],
        [37.793,-122.397],[37.789,-122.401],[37.784,-122.408],[37.780,-122.415],
        [37.765,-122.419],[37.752,-122.419],[37.733,-122.434],[37.722,-122.447],
        [37.706,-122.469],
    ])
    CALTRAIN_STATIONS = np.array([
        [37.776,-122.394],[37.758,-122.392],[37.709,-122.401],[37.654,-122.444],
        [37.630,-122.411],[37.600,-122.387],[37.580,-122.345],[37.567,-122.323],
        [37.553,-122.309],[37.537,-122.296],[37.521,-122.276],[37.508,-122.260],
        [37.486,-122.232],[37.464,-122.198],[37.454,-122.182],[37.444,-122.165],
        [37.430,-122.143],[37.407,-122.107],[37.394,-122.077],[37.378,-122.031],
        [37.371,-121.997],[37.353,-121.936],[37.330,-121.902],[37.312,-121.883],
        [37.133,-121.654],[37.086,-121.613],[37.005,-121.567],
    ])
    LA_METRO_BLUE = np.array([
        [34.048,-118.259],[34.037,-118.265],[34.026,-118.268],[34.018,-118.267],
        [33.979,-118.269],[33.975,-118.266],[33.964,-118.261],[33.957,-118.253],
        [33.940,-118.245],[33.921,-118.239],[33.898,-118.221],[33.866,-118.205],
        [33.843,-118.197],[33.823,-118.200],[33.804,-118.191],[33.769,-118.190],
    ])
    bart_d  = np.column_stack([haversine_distance(lats, lons, s[0], s[1]) for s in BART_STATIONS])
    cal_d   = np.column_stack([haversine_distance(lats, lons, s[0], s[1]) for s in CALTRAIN_STATIONS])
    metro_d = np.column_stack([haversine_distance(lats, lons, s[0], s[1]) for s in LA_METRO_BLUE])
    combined['dist_nearest_bart']     = bart_d.min(axis=1)
    combined['dist_nearest_caltrain'] = cal_d.min(axis=1)
    combined['dist_nearest_metro']    = metro_d.min(axis=1)
    combined['dist_nearest_transit']  = np.minimum(
        np.minimum(combined['dist_nearest_bart'], combined['dist_nearest_caltrain']),
        combined['dist_nearest_metro']
    )
    combined['in_transit_walkzone']  = (combined['dist_nearest_transit'] < 2.0).astype(int)
    combined['in_bart_walkzone']     = (combined['dist_nearest_bart']    < 2.0).astype(int)
    combined['transit_access_score'] = 1.0 / (combined['dist_nearest_transit'] + 1.0)

    # 18. 緯度・経度PCA
    pca = PCA(n_components=2, random_state=42)
    train_coords = combined.loc[:n_train-1, ['Latitude', 'Longitude']]
    pca.fit(train_coords)
    pca_coords = pca.transform(combined[['Latitude', 'Longitude']])
    combined['geo_pca1']      = pca_coords[:, 0]
    combined['geo_pca2']      = pca_coords[:, 1]
    combined['lat_lon_inter'] = combined['Latitude'] * combined['Longitude']

    # 19. Geo-cluster KMeans
    km = KMeans(n_clusters=50, random_state=42, n_init=10)
    km.fit(train_coords)
    combined['geo_cluster'] = km.predict(combined[['Latitude', 'Longitude']])

    # 20. 防衛産業クラスター (1990年・冷戦末期の主要軍需工場)
    # 南カリフォルニアは全米最大の防衛産業集積地。冷戦終結後の削減前で雇用が最大規模
    DEFENSE_PLANTS = {
        'Lockheed_Palmdale':       (34.58, -118.08),  # Skunk Works: SR-71/F-117製造
        'Lockheed_Burbank':        (34.18, -118.33),  # 旧Skunk Works拠点
        'Lockheed_Sunnyvale':      (37.39, -122.03),  # ミサイル・宇宙部門
        'Northrop_Hawthorne':      (33.92, -118.33),  # B-2爆撃機開発中
        'McDonnell_Douglas_LB':    (33.82, -118.19),  # C-17輸送機製造
        'Raytheon_ElSegundo':      (33.92, -118.40),  # ミサイルシステム
        'Rockwell_Downey':         (33.94, -118.13),  # スペースシャトル製造
        'Hughes_Aircraft_Culver':  (34.00, -118.40),  # レーダー・衛星システム
        'TRW_Redondo':             (33.86, -118.37),  # 宇宙・防衛電子
        'General_Dynamics_Pomona': (34.06, -117.75),  # ミサイルシステム
        'Aerojet_Sacramento':      (38.55, -121.38),  # ロケットエンジン
        'Lawrence_Livermore':      (37.69, -121.70),  # 核兵器研究所（高給雇用）
        'Loral_SanJose':           (37.33, -121.89),  # 防衛電子
        'Rockwell_Seal_Beach':     (33.74, -118.10),  # スペースシャトル部品
    }
    def_cols = []
    for plant, (plat, plon) in DEFENSE_PLANTS.items():
        col = f'dist_def_{plant}'
        combined[col] = haversine_distance(lats, lons, plat, plon)
        def_cols.append(col)
    combined['dist_nearest_defense']   = combined[def_cols].min(axis=1)
    combined['defense_cluster_score']  = 1.0 / (combined['dist_nearest_defense'] + 1.0)
    combined['defense_top3_mean_dist'] = combined[def_cols].apply(
        lambda r: np.mean(sorted(r.values)[:3]), axis=1
    )

    # 21. 大気汚染スコア (1990年 CARB オゾン超過日数ベースのスモッグ指数)
    # 南海岸（LAベイシン）・サンホアキン渓谷が全米最悪レベル
    AIR_QUALITY_REFS = pd.DataFrame({
        'lat': [
            33.98, 34.11, 35.37, 36.75, 34.09,   # 最悪 (Riverside/SB/Bakersfield/Fresno/Fontana)
            34.05, 34.15, 34.07, 33.77, 34.18,   # 中程度 (LAベイシン中心部)
            34.02, 37.77, 37.80, 37.34,           # やや良 (海岸・SF湾)
            34.42, 32.72, 36.60, 38.58, 40.80,   # 良 (北部・沿岸)
        ],
        'lon': [
            -117.37, -117.29, -119.02, -119.77, -117.43,
            -118.24, -118.14, -118.03, -118.19, -118.31,
            -118.49, -122.42, -122.27, -121.89,
            -119.70, -117.16, -121.90, -121.49, -124.16,
        ],
        'smog_score': [
            10, 9, 10, 9, 9,
             7, 8,  8, 7, 7,
             4, 3,  4, 4,
             3, 4,  2, 5, 1,
        ]
    })
    aq_tree = KDTree(AIR_QUALITY_REFS[['lat', 'lon']].values)
    aq_dists, aq_idx = aq_tree.query(combined[['Latitude', 'Longitude']].values, k=3)
    aq_w = 1.0 / (aq_dists + 1e-3)
    aq_w /= aq_w.sum(axis=1, keepdims=True)
    combined['smog_score_1990'] = (AIR_QUALITY_REFS['smog_score'].values[aq_idx] * aq_w).sum(axis=1)
    # 海岸距離との交互作用（海に近いほど実際のスモッグは和らぐ）
    combined['smog_x_coast'] = combined['smog_score_1990'] * combined['coast_dist_precise']

    feature_cols = [c for c in combined.columns if c not in ['id']]
    X_train_fe = combined.iloc[:n_train][feature_cols].reset_index(drop=True)
    X_test_fe  = combined.iloc[n_train:][feature_cols].reset_index(drop=True)
    return X_train_fe, y_tr, X_test_fe

X, y, X_test = add_features(train, test)
print(f'特徴量生成完了: {X.shape[1]} カラム')
new_cols = [c for c in X.columns if c not in
    ['id','MedInc','HouseAge','AveRooms','AveBedrms','Population','AveOccup','Latitude','Longitude','Household','AllRooms','AllBedrms']]
print(f'追加特徴量 ({len(new_cols)}個):', new_cols)

特徴量生成完了: 108 カラム
追加特徴量 (97個): ['rooms_per_person', 'bedrooms_ratio', 'income_per_room', 'rooms_per_household', 'bedrms_per_household', 'total_income_proxy', 'nonbedroom_rooms', 'household_density', 'dist_LA', 'dist_SF', 'dist_SD', 'dist_SJ', 'dist_SB', 'dist_nearest_city', 'coast_dist_proxy', 'dist_to_elite_univ', 'nearest_univ_brand_score', 'univ_influence_score', 'nearest_crime_idx_1990', 'dist_to_crime_benchmark', 'local_crime_exposure', 'dist_san_andreas', 'dist_hayward_fault', 'min_fault_dist', 'seismic_risk_score', 'dist_mil_NAS_North_Island', 'dist_mil_NAS_Alameda', 'dist_mil_Travis_AFB', 'dist_mil_Edwards_AFB', 'dist_mil_Vandenberg_AFB', 'dist_mil_Fort_Ord', 'dist_mil_MCAS_El_Toro', 'dist_mil_Point_Mugu', 'dist_mil_Castle_AFB', 'dist_mil_NAS_Miramar', 'dist_nearest_military', 'dist_port_LA_LB', 'dist_port_Oakland', 'dist_port_SD', 'dist_nearest_port', 'dist_silicon_valley', 'dist_mexico_border', 'dist_aff_Beverly_Hills', 'dist_aff_Palo_Alto', 'dist_aff_Marin', 'dist_aff_Newport

### 特徴量とPriceの相関係数

In [ ]:
# Price（対数変換前）との相関係数を絶対値降順で表示
price_raw = np.expm1(y)
corr_series = X.apply(lambda col: col.corr(price_raw))
corr_df = corr_series.abs().sort_values(ascending=False).rename('|corr|').to_frame()
corr_df['corr'] = corr_series[corr_df.index]
corr_df = corr_df[['corr', '|corr|']]
print(corr_df.to_string())

                                      corr    |corr|
income_x_county_edu               0.701062  0.701062
MedInc                            0.689659  0.689659
income_per_room                   0.662105  0.662105
dist_nearest_affluent            -0.508540  0.508540
coast_dist_precise               -0.494301  0.494301
geo_pca2                         -0.490935  0.490935
smog_x_coast                     -0.481301  0.481301
dist_nearest_city                -0.456567  0.456567
dist_to_elite_univ               -0.440086  0.440086
dist_nearest_transit             -0.435612  0.435612
affluent_gravity                  0.427698  0.427698
coast_dist_proxy                 -0.423131  0.423131
dist_nearest_port                -0.419888  0.419888
defense_top3_mean_dist           -0.406273  0.406273
income_x_la                       0.382862  0.382862
income_x_bay                      0.369336  0.369336
county_edu_score                  0.356157  0.356157
dist_nearest_defense             -0.352874  0.

### 5-Fold 交差検証と対数変換を用いたモデルの学習・評価

In [ ]:
KF = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
import subprocess
try:
  subprocess.check_output('nvidia-smi', stderr=subprocess.DEVNULL)
  xgb_device = 'cuda'
  print("GPU検出: XGBoost は cuda で実行します")
except Exception:
  xgb_device = 'cpu'
  print("GPU未検出: XGBoost は cpu で実行します")
def objective_xgb(trial):
    params = {
        'n_estimators': 3000,
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 2.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 2.0, log=True),
        'random_state': 42,
        'tree_method': 'hist',
        'n_jobs': -1,
        'device': xgb_device,
    }

    scores = []
    for tr_idx, va_idx in KF.split(X):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )
        # XGBoostのデフォルトのEarly Stopping（パラメータ内に仕込むかfitで制御）
        # ※ここではOptuna用にシンプルな評価
        preds = np.expm1(model.predict(X_va))
        actuals = np.expm1(y_va)
        scores.append(np.sqrt(mean_squared_error(actuals, preds)))

    return np.mean(scores)

study_xgb = optuna.create_study(direction='minimize')
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)
print(f'parameter_set : {study_xgb.best_params}')
print(f'Best XGBoost CV (RMSE): {study_xgb.best_value:.4f}')

GPU検出: XGBoost は cuda で実行します


  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-05-24 14:12:52,182] Trial 0 failed with parameters: {'max_depth': 10, 'learning_rate': 0.022730558388138235, 'subsample': 0.6844596376465198, 'colsample_bytree': 0.901337847989983, 'reg_alpha': 0.9482177922297117, 'reg_lambda': 0.05664196834641409} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_57/2829533350.py", line 30, in objective_xgb
    model.fit(
  File "/usr/local/lib/python3.12/dist-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/sklearn.py", line 1368, in fit
    self._Booster = train(
                    ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
           

KeyboardInterrupt: 

In [ ]:
xgb_device='cuda'
best_params_xgb = {
      'max_depth':        8,
      'learning_rate':    0.01661220560195241,
      'subsample':        0.9157632062844006,
      'colsample_bytree': 0.602181434610246,
      'reg_alpha':        0.0023806589316786608,
      'reg_lambda':       0.028903948082561075,
  }

  # 使う時はこう
params = {
      **best_params_xgb,
      'n_estimators': 5000,
      'random_state': 42,
      'tree_method': 'hist',
      'device':xgb_device,
  }
model = XGBRegressor(**params, early_stopping_rounds=50)
# best_params_cat = {
#     # max_depth=8 → depth はそのまま対応
#     'depth': 8,

#     # learning_rate=0.0166 → CatBoostは少し高めが効きやすい
#     'learning_rate': 0.02,

#     # subsample=0.916（高め＝ランダム性少）→ bagging_temperature は低めに
#     # Bernoulliモードで subsample を直接指定する方が直感的
#     'bootstrap_type': 'Bernoulli',
#     'subsample': 0.9,

#     # colsample_bytree=0.602 → CatBoostではrsm（Random Subspace Method）
#     'rsm': 0.6,

#     # reg_alpha≈0, reg_lambda≈0.029（非常に小さい）→ 正則化は軽め
#     'l2_leaf_reg': 2.0,

#     # XGBの正則化が弱かったので random_strength も小さめ
#     'random_strength': 0.5,
# }

# params_cat = {
#     **best_params_cat,
#     'iterations': 5000,
#     'random_seed': 42,
#     'verbose': 0,
#     'task_type': 'GPU',
#     'cat_features': cat_features_idx,
#     'od_type': 'Iter',
#     'od_wait': 50,
# }


In [ ]:
cat_features_idx = [X.columns.get_loc('geo_cluster')]

def objective_cat(trial):
    params = {
        'iterations': 3000,
        'task_type': 'GPU',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 6, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 3.0, 12.0),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_seed': 42,
        'verbose': 0,
        'cat_features': cat_features_idx,
    }
    scores = []
    for tr_idx, va_idx in KF.split(X):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
        model = CatBoostRegressor(**params)
        model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)
        preds = np.expm1(model.predict(X_va))
        scores.append(np.sqrt(mean_squared_error(np.expm1(y_va), preds)))
    return np.mean(scores)

TARGET_RMSE = 0.414
MAX_TRIALS = 20  # 無限ループ防止の上限

def stop_if_target_reached(study, trial):
    if study.best_value < TARGET_RMSE:
        study.stop()

study_cat = optuna.create_study(direction='minimize')
study_cat.optimize(
    objective_cat,
    n_trials=MAX_TRIALS,
    callbacks=[stop_if_target_reached],
    show_progress_bar=True
)

print(f'parameter_set : {study_cat.best_params}')
print(f'Best CatBoost CV (RMSE): {study_cat.best_value:.4f}')

if study_cat.best_value < TARGET_RMSE:
    print(f"✅ 目標 {TARGET_RMSE} 達成！({len(study_cat.trials)} trials)")
else:
    print(f"⚠️ {MAX_TRIALS} trials 到達したが未達成 (best: {study_cat.best_value:.4f})")

# ── OOF格納用を初期化（セル単独実行でも動くよう先頭で定義）──
oof_preds  = pd.DataFrame(index=X.index)
test_preds = pd.DataFrame(index=X_test.index)

# --- 1. XGBoost OOF ---
xgb_oof  = np.zeros(len(X))
xgb_test = np.zeros(len(X_test))

for tr_idx, va_idx in KF.split(X):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx] #study_xgb.best_params(学習時)
    params = {**best_params_xgb, 'n_estimators': 5000, 'random_state': 42,
              'tree_method': 'hist', 'device': xgb_device}
    model = XGBRegressor(**params, early_stopping_rounds=50)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    xgb_oof[va_idx] = np.expm1(model.predict(X_va))
    xgb_test += np.expm1(model.predict(X_test)) / KF.n_splits

oof_preds['xgb']  = xgb_oof
test_preds['xgb'] = xgb_test
print(f"XGB OOF RMSE: {np.sqrt(mean_squared_error(np.expm1(y), xgb_oof)):.4f}")
print(f"parameter_set : {model.get_params()}")

# --- 2. CatBoost OOF ---
cat_oof  = np.zeros(len(X))
cat_test = np.zeros(len(X_test))

for tr_idx, va_idx in KF.split(X):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]
    model = CatBoostRegressor( #study_cat.best_params
        **study_cat.best_params, iterations=5000,
        random_seed=42, verbose=0, cat_features=cat_features_idx,
    )
    model.fit(X_tr, y_tr, eval_set=(X_va, y_va), early_stopping_rounds=50)
    cat_oof[va_idx] = np.expm1(model.predict(X_va))
    cat_test += np.expm1(model.predict(X_test)) / KF.n_splits

oof_preds['cat']  = cat_oof
test_preds['cat'] = cat_test
print(f"CatBoost OOF RMSE: {np.sqrt(mean_squared_error(np.expm1(y), cat_oof)):.4f}")
print(f"parameter_set : {model.get_params()}")

  0%|          | 0/16 [00:00<?, ?it/s]

### 提出用ファイルの作成と保存

In [ ]:
from scipy.optimize import minimize

# 元の価格スケールでの正解ラベル
actual_prices = np.expm1(y)

# XGB + CatBoost の2モデルで重みを最適化
def rmse_func(weights):
    blend = (weights[0] * oof_preds['xgb'] +
             weights[1] * oof_preds['cat'])
    return np.sqrt(mean_squared_error(actual_prices, blend))

bounds = [(0.0, 1.0), (0.0, 1.0)]
res = minimize(rmse_func, [0.5, 0.5], method='L-BFGS-B', bounds=bounds)
best_weights = res.x / (np.sum(res.x) + 1e-9)

print("── 最適アンサンブル比率 ──")
print(f"  XGBoost  : {best_weights[0]:.3f}")
print(f"  CatBoost : {best_weights[1]:.3f}")

final_oof = (best_weights[0] * oof_preds['xgb'] +
             best_weights[1] * oof_preds['cat'])
print(f"\nFinal Ensemble OOF RMSE: {np.sqrt(mean_squared_error(actual_prices, final_oof)):.5f}")

final_test_preds = (best_weights[0] * test_preds['xgb'] +
                    best_weights[1] * test_preds['cat'])

final_test_preds = np.clip(final_test_preds, 0.14999, 5.00001)

sample['Price'] = final_test_preds
sample.to_csv('submit_improved_final.csv', index=False)
print('\n提出ファイル作成完了: submit_improved_final.csv')

In [ ]:
sample_cv_advanced = sample.copy()
sample_cv_advanced['Price'] = final_test_preds
output_path = '/submit_improved_final.csv'
sample_cv_advanced.to_csv(output_path, index=None)

print(f"最終提出用ファイル '{output_path}' を作成しました。Kaggleへの提出準備は万端です！")

### Kaggleへの直接投稿

In [ ]:
# 作成したファイルをKaggleに直接投稿
!kaggle competitions submit -c ambl-california-housing -f /submit_improved_final.csv -m "step=16_and_depth add bouei,taiki"